In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(0)

In [ ]:
df = pd.read_excel("/kaggle/input/datasets/lordmostafa199/arabizi-arabic-parallel-corpora/Arabizi-Arabic Parallel corpora.xlsx")
df.head()

## Fixing the Dataset

In [ ]:
# Replace specific numbers with letters in the text column
df['Arabize'] = (
    df['Arabize']
    .astype(str)
    .str.replace('8', 'k', regex=False)
    .str.replace('6', 't', regex=False)
    .str.replace('9', 's', regex=False)
)
df.head(10)

In [ ]:
print(df.shape)
df.info()

In [ ]:
# Set device to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 1) Load the 20,000 rows
sub = df.iloc[:20000]
pairs = list(zip(sub["Arabize"], sub["Arabic"]))

PAD, SOS, EOS = 0, 1, 2


# Vocabulary mapping characters to integers
def make_vocab(strs):
  return {c: i + 3 for i, c in enumerate(sorted(set("".join(strs))))}


src_v = make_vocab([a for a, _ in pairs])
tgt_v = make_vocab([b for _, b in pairs])
inv_tgt = {i: c for c, i in tgt_v.items()}


# Create a Custom PyTorch Dataset
class ArabiziDataset(Dataset):

  def __init__(self, pairs, src_vocab, tgt_vocab):
    self.pairs = pairs
    self.src_vocab = src_vocab
    self.tgt_vocab = tgt_vocab

  def __len__(self):
    return len(self.pairs)

  def __getitem__(self, idx):
    src_str, tgt_str = self.pairs[idx]
    src_ids = [self.src_vocab[c] for c in src_str if c in self.src_vocab]
    tgt_ids = (
        [SOS] + [self.tgt_vocab[c] for c in tgt_str if c in self.tgt_vocab] + [EOS]
    )
    return torch.tensor(src_ids, dtype=torch.long), torch.tensor(
        tgt_ids, dtype=torch.long
    )


# Collate function to handle dynamic padding per batch
def collate_fn(batch):
  srcs, tgts = zip(*batch)

  # Pad source sequences
  max_src_len = max(len(s) for s in srcs)
  padded_srcs = torch.full((len(srcs), max_src_len), PAD, dtype=torch.long)
  for i, s in enumerate(srcs):
    padded_srcs[i, : len(s)] = s

  # Pad target sequences
  max_tgt_len = max(len(t) for t in tgts)
  padded_tgts = torch.full((len(tgts), max_tgt_len), PAD, dtype=torch.long)
  for i, t in enumerate(tgts):
    padded_tgts[i, : len(t)] = t

  return padded_srcs, padded_tgts


# Initialize DataLoader with a safe batch size (e.g., 64)
dataset = ArabiziDataset(pairs, src_v, tgt_v)
dataloader = DataLoader(
    dataset, batch_size=64, shuffle=True, collate_fn=collate_fn
)


# 2) Model Definition
class Seq2Seq(nn.Module):

  def __init__(s, nsrc, ntgt, emb=64, hid=256, layers=4):
    super().__init__()
    s.se = nn.Embedding(nsrc, emb, padding_idx=PAD)
    s.te = nn.Embedding(ntgt, emb, padding_idx=PAD)
    s.enc = nn.LSTM(emb, hid, layers, batch_first=True)
    s.dec = nn.LSTM(emb, hid, layers, batch_first=True)
    s.out = nn.Linear(hid, ntgt)

  def forward(s, x, y_in):
    _, st = s.enc(s.se(x))
    o, _ = s.dec(s.te(y_in), st)
    return s.out(o)


V = len(tgt_v) + 3
m = Seq2Seq(len(src_v) + 3, V).to(device)

opt = torch.optim.Adam(m.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss(ignore_index=PAD)

# 3) Mini-batch Training Loop
epochs = 10  # With 20k rows, 10-20 epochs are usually plenty for a baseline
for ep in range(1, epochs + 1):
  m.train()
  total_loss = 0

  for X_batch, Y_batch in dataloader:
    X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

    logits = m(X_batch, Y_batch[:, :-1])
    loss = lossf(logits.reshape(-1, V), Y_batch[:, 1:].reshape(-1))

    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    opt.step()

    total_loss += loss.item()

  avg_loss = total_loss / len(dataloader)
  print(f"Epoch {ep}/{epochs}, Loss: {round(avg_loss, 4)}")

In [ ]:
# Extract the test pairs from the remaining dataset
test_sub = df.iloc[20000:]
test_pairs = list(zip(test_sub["Arabize"], test_sub["Arabic"]))


@torch.no_grad()
def evaluate_test_set(pairs):
  m.eval()
  correct = 0
  total = len(pairs)

  for src_text, target_text in pairs:
    # Convert source text to tensor
    src_ids = [src_v[c] for c in src_text if c in src_v]
    if not src_ids:
      continue
    x = torch.tensor([src_ids], dtype=torch.long, device=device)

    # Greedy decoding for a single sample
    _, st = m.enc(m.se(x))
    tok = torch.full((1, 1), SOS, dtype=torch.long, device=device)
    outs = []

    for _ in range(len(src_text) * 2):  # Dynamic max length based on input size
      o, st = m.dec(m.te(tok), st)
      tok = m.out(o).argmax(-1)
      if tok.item() == EOS:
        break
      outs.append(tok.item())

    # Convert predicted IDs back to string
    pred_str = "".join([inv_tgt[i] for i in outs if i in inv_tgt])

    if pred_str == target_text:
      correct += 1

  accuracy = correct / total if total > 0 else 0
  print(f"Test Exact-Match Accuracy: {accuracy * 100:.2f}%")


# Run the evaluation
evaluate_test_set(test_pairs)

In [ ]:
@torch.no_grad()
def translate_sentence_word_by_word(sentence):
    m.eval()

    # 1. Split the sentence into words using spaces
    words = sentence.split(" ")
    translated_words = []

    for word in words:
        # If the word is empty, preserve the space
        if not word.strip():
            translated_words.append(" ")
            continue

        # 2. Convert the word's characters into model input IDs
        src_ids = [src_v[c] for c in word if c in src_v]

        if not src_ids:
            # If no characters are found in the vocabulary, keep the original word
            translated_words.append(word)
            continue

        x = torch.tensor(
            [src_ids],
            dtype=torch.long,
            device=device
        )

        # 3. Encode the word and initialize the decoder
        _, st = m.enc(m.se(x))
        tok = torch.full(
            (1, 1),
            SOS,
            dtype=torch.long,
            device=device
        )
        decoded_ids = []

        # Generate the Arabic characters one by one
        for _ in range(len(word) * 2):
            o, st = m.dec(m.te(tok), st)
            tok = m.out(o).argmax(-1)

            predicted_id = tok.item()

            # Stop generating when the end-of-sequence token is predicted
            if predicted_id == EOS:
                break

            decoded_ids.append(predicted_id)

        # 4. Convert the predicted IDs back into characters
        pred_word = "".join([
            inv_tgt[i] for i in decoded_ids if i in inv_tgt
        ])

        translated_words.append(pred_word)

    # 5. Join the translated words with spaces
    return " ".join(translated_words)

In [ ]:
@torch.no_grad()
def translate_arabizi_sentence(sentence):
  m.eval()

  # Map characters to source vocabulary IDs (ignore characters the model hasn't seen)
  src_ids = [src_v[c] for c in sentence if c in src_v]
  if not src_ids:
    return "Error: Input contains characters unknown to the model vocabulary."

  # Convert to tensor and send to device
  x = torch.tensor([src_ids], dtype=torch.long, device=device)

  # Encode
  _, st = m.enc(m.se(x))

  # Decode step-by-step using greedy search
  tok = torch.full((1, 1), SOS, dtype=torch.long, device=device)
  decoded_ids = []

  # Allow up to 2x the input length to complete the translation
  max_length = len(sentence) * 2
  for _ in range(max_length):
    o, st = m.dec(m.te(tok), st)
    tok = m.out(o).argmax(-1)

    predicted_id = tok.item()
    if predicted_id == EOS:
      break

    decoded_ids.append(predicted_id)

  # Convert predicted token IDs back to an Arabic string
  translated_sentence = "".join([inv_tgt[i] for i in decoded_ids if i in inv_tgt])
  return translated_sentence


# Test your inference engine with a sentence!
sample_input = "salam 3amel eh"
print(f"Input Arabizi: {sample_input}")
print(f"Translated Arabic: {translate_arabizi_sentence(sample_input)}")


In [ ]:
print(f"{'Input (Arabizi)':<20} | {'Target (Arabic)':<20} | {'Prediction':<20}")
print("-" * 65)

# Take the first 20 pairs from your test set
for src_text, target_text in test_pairs[:20]:
  pred_str = translate_arabizi_sentence(src_text)
  print(f"{src_text:<20} | {target_text:<20} | {pred_str:<20}")